# Exercise 3.4.10 — Let the LLM see its entire chat history

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `3.4 LLM Agents`  
**Notebook:** `3.4_LLM_Agents_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=3.4.10](https://delta-drills.vercel.app/?arena_exercise=3.4.10)


# [3.4] - LLM Agents (exercises)

> **ARENA [Streamlit Page](https://arena-chapter3-llm-evals.streamlit.app/04_[3.4]_LLM_Agents)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter3_llm_evals/exercises/part4_llm_agents/3.4_LLM_Agents_exercises.ipynb?t=20260303) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter3_llm_evals/exercises/part4_llm_agents/3.4_LLM_Agents_solutions.ipynb?t=20260303)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src = "https://raw.githubusercontent.com/callummcdougall/computational-thread-art/master/example_images/misc/robot-typewriter.png" width = "350">

# Introduction

This set of exercises can last up to 2 days, and involves building and working with LLM agents using Inspect. LLM agents consist of a scaffolding program interacting with an LLM API. We'll also build and analyse two tasks for LLM agents to complete, a simple and a complex one, in order to see how LLM agents act.

We'll begin by building a simple Arithmetic Task and Arithmetic Agent. This should teach you the basics of function calling using Inspect. Then, once we're comfortable with function calling and the general setup of LLM agents and tasks, we'll move on to building a more complex agent that plays the [Wikipedia Game](https://en.wikipedia.org/wiki/Wikipedia:Wiki_Game).

Then we'll explore a variety of elicitation methods. These are methods for getting the best capabilities out of models, and are crucial for evaluating LLM agents. Looking at model performance with elicitation method help us to answer the question "Can the model do this?" Unfortunately, we'll almost never be able to *prove* that the model doesn't have a capability, and will only be able to say that *with some effort*, we couldn't get the model to demonstrate this capability. This means we'll have to put a lot of effort into trying to exhibit the behavior in models (to have the highest confidence when we make a claim that the model can't exhibit this behavior). This will involve:

* Improving our prompting
* Improving our tools
* Improving the way the relevant information is stored
* Ensuring the model can access good information

Each exercise will have a difficulty and importance rating out of 5, as well as an estimated maximum time you should spend on these exercises and sometimes a short annotation. You should interpret the ratings & time estimates relatively (e.g. if you find yourself spending about 50% longer on the exercises than the time estimates, adjust accordingly). Please do skip exercises / look at solutions if you don't feel like they're important enough to be worth doing, and you'd rather get to the good stuff!

For a lecture on the material today, which provides some high-level understanding before you dive into the material, watch the video below:

<iframe width="540" height="304" src="https://www.youtube.com/embed/H7hXqm1idAI" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

## Content & Learning Objectives

### 1️⃣ Intro to LLM Agents

> ##### Learning Objectives
> - Understand why we want to evaluate LLM agents.
> - Read resources about LLM agent evaluations to understand the current state of the field.
> - Understand the common failure modes of LLM agents.

### 2️⃣ Building a Simple Arithmetic Agent

> ##### Learning Objectives
> - Understand that a LLM agent is just a "glorified for-loop" (of the scaffolding program interacting with the LLM API).
> - Learn how to use function calling to allow LLMs to use external tools.
> - Understand the main functionalities of an LLM agent.

### 3️⃣ Building a more Complex Agent: WikiGame

> ##### Learning Objectives
> - Get comfortable building a more complex task, with noisy and imperfect tool outputs
> - Understand how to build a more complex agent that implements dynamic decision-making
> - Observe the failure modes of a more complex agent

### 4️⃣ Elicitation

> ##### Learning Objectives
> - Understand the importance of elicitation in evaluating LLM agents
> - Understand the different methods of elicitation
> - Understand how to improve prompting, tools, history storage, and information access in LLM agents

In [ ]:
import os
import sys
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter3_llm_evals"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import inspect_ai
except:
    %pip install openai>=1.56.1 anthropic inspect_ai tabulate wikipedia jaxtyping

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}

if IN_COLAB:
    from google.colab import userdata

    try:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except:
        warnings.warn(
            "You don't have an OPENAI_API_KEY variable set in the secrets tab of your google colab. You have to set one, or any calls to APIs won't work."
        )


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import os
import re
import sys
from pathlib import Path
from typing import Literal

import wikipedia
from dotenv import load_dotenv
from inspect_ai import Task, eval, task
from inspect_ai.agent import Agent, AgentState, agent, as_solver
from inspect_ai.dataset import Sample
from inspect_ai.model import (
    ChatMessageSystem,
    ChatMessageTool,
    ChatMessageUser,
    execute_tools,
    get_model,
)
from inspect_ai.tool import Tool, tool
from openai import OpenAI
from wikipedia import DisambiguationError, PageError, WikipediaPage

# Make sure exercises are in the path
chapter = "chapter3_llm_evals"
section = "part4_llm_agents"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_llm_agents.tests as tests
from part4_llm_agents.utils import evaluate_expression, execute_tools, extract_answer

EVAL_MODEL = "openai/gpt-4o-mini"
os.environ["INSPECT_EVAL_MODEL"] = EVAL_MODEL
MAIN = __name__ == "__main__"

In [ ]:
assert os.getenv("OPENAI_API_KEY") is not None, "You must set your OpenAI API key - see instructions in dropdown"

# OPENAI_API_KEY

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 1️⃣ Intro to LLM Agents

> ##### Learning Objectives
>
> - Read resources about LLM agent evaluations to understand the current state of the field.
> - Understand the common failure modes of LLM agents.

## What is an LLM agent?

An LLM agent consists of a scaffolding program interacting with an LLM API to accomplish tasks in an external environment. This typically involves a loop of the following steps:

1. The scaffolding program sends instructions to the LLM, typically containing information about the task goal, the possible actions available to the LLM, and any other relevant task information. (e.g. you are trying to calculate `3+3`)
2. The LLM processes the input and outputs an action in text (e.g. "calls" the `calculate()` tool on the expression `3+3` in text).
3. The scaffolding program executes the action and returns the outcome (e.g. it runs the `calculate()` function in the background and returns the output, `6`, to the agent).
4. The LLM observes the results and decides the next action.
5. Repeating the cycle until the task is complete.

The two basic components of scaffolding are:

* Tool calling: This allows LLMs to use tools by providing a text description of the tool. The LLM can choose to use this tool by "calling" it in its text output. If it uses a tool, the scaffolding will execute this tool on the LLM's behalf (e.g. by running a python function, sending request to an external API etc.) and return the result of this tool call to the agent.
* Prompting: This describes the task state to the LLM, describes the tools available to the LLM in the task, potentially instructs the LLM to use chain-of-thought to give the LLM more "thinking time." This also covers how the LLM's `chat_history` is stored from the LLM's prior actions.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/ch3-llm-agent.png" width="800">

Diagram based on METR's [*Evaluating Language-Model Agents on Realistic Autonomous Tasks*](https://arxiv.org/abs/2312.11671), Figure 2.

## Why evaluate LLM agents?

There are at least two reasons we want to evaluate LLM agents.

1. **Measuring the maximum capabilities of a model**

For estimating safety risks, we want to measure the **ceiling** of dangerous capabilities. LLMs on their own often fail in easy-to-fix ways, as you will see. For example:

- They often claim to be incapable of tasks that they can actually perform.
- They can *very* easily get stuck in loops.
- They can give up and ask the user for help
- They can hallucinate facts, or even misunderstand their own prior reasoning and hallucinate a faulty conclusion.
- They can be limited by primitive tools.
- They can be sensitive in strange ways to information in their prompts.
- They can have bugs, typos, or other minor barriers that prevent them from operating to the fullest extent of their capability.

This means that when a model fails to accomplish a task, it may still have the capability to succeed; requiring only simple fixes that will unlock this capability. We want to eliminate the possibility of large capability improvements from relatively little effort, because this means our evaluation would have underestimated the true capability and risks associated with a model (especially in e.g. a dangerous capabilities evaluation). Therefore, we to try hard to elicit their raw capabilities (e.g. using scaffolding), so that we can evaluate LLMs at their *best*.


2. **Measuring the alignment of LLMs in agentic scenarios**

We do not know if our current alignment techniques (e.g. supervised fine-tuning, RLHF) for aligning LLM chatbots will still work when LLMs are acting as agents in more complex scenarios. It is possible that these methods will not generalize well to agentic scenarios, and we may want to test this.

We know today that LLMs are being used as more than just chatbots. Since the release of ChatGPT, the use of LLMs as agentic systems has grown signifcantly. These agents started off rather disappointingly, when they were based on GPT-3.5. However as more powerful LLMs come out and AI companies ensure their LLMs are better at tool-use, these agents are improving rapidly.

<details><summary>Further resources on LLM agent evaluations</summary>

- [Evaluating Language-Model Agents on Realistic Autonomous Tasks](https://evals.alignment.org/Evaluating_LMAs_Realistic_Tasks.pdf) (Kinniment et al., ARC Evaluations Team (now METR), 2023)
- [Large Language Models can Strategically Deceive their Users when Put Under Pressure](https://arxiv.org/pdf/2311.07590) (Scheurer et al., Apollo Research, ICLR 2024)
- [LLM Powered Autonomous Agents](https://lilianweng.github.io/posts/2023-06-23-agent/) (Lilian Weng, OpenAI Safety Team, 2023)
- [AXRP Episode 34 - AI Evaluations with Beth Barnes](https://www.alignmentforum.org/posts/vACr4DExfeRMaCoo7/axrp-episode-34-ai-evaluations-with-beth-barnes) (Daniel Filan, 2024)
-[Reflexion: Language Agents with Verbal Reinforcement Learning](https://arxiv.org/pdf/2303.11366) (Shinn et al., 2023)
- [Answering Questions by Meta-Reasoning over Multiple Chains of Thought](https://arxiv.org/pdf/2304.13007) (Yoran et al., 2024)
- [Toolformer: Language Models Can Teach Themselves to Use Tools](https://arxiv.org/pdf/2302.04761) (Schick et al., META AI Research, 2023)
- [OpenAI Function Calling Guide](https://platform.openai.com/docs/guides/function-calling)
- [Anthropic Function Calling Guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)

</details>

# 2️⃣ Building a Simple Arithmetic Agent

> ##### Learning Objectives
>
> - Learn how to use function calling to allow LLMs to use external tools.
> - Understand the main functionalities of an LLM agent.

In general, most LLM agents share these core components:

<img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/refs/heads/main/img/ch3-sec4-agent-overview.png" width="1000">

1. **LLM API interface**: A basic function that makes API calls (e.g. `generate()`). <!-- (IN AGENT)-->
2. **Actions/Tools**: A set of actions the agent can take. <!-- (MOSTLY IN TASK)-->
3. **Task State Management**: Keeping track of the current state of the task and any relevant context. <!-- (IN TASK MOSTLY)-->
4. **Memory**: A way to store and retrieve information from past interactions (i.e. chat history). In inspect, we store it in `state.messages`. <!-- (IN AGENT)-->
5. **Observation Parser**: Functions to parse and interpret the results of actions and update the state. <!-- (IN TASK/TOOLS MOSTLY)-->
6. **Decision/Execution Logic**: The rules or algorithms used to choose actions based on the current state and LLM output. <!-- (MOSTLY IN AGENT)-->
7. **Task-Specific Information**: Any additional information or functions specific to the task at hand. <!-- (INFO IN AGENT/FUNCTIONS IN TASK)-->

These components are implemented across the `Task`, `Agent`, and `Tool` functions/classes. However, the specific breakdown of these components in our implementation is a design choice and can vary depending on the task. While some are very natural (e.g. LLM API interface goes into `Agent`, task state management goes into `Task`), others can vary (e.g. `Tool`s could be implemented and handled entirely within the `Task` or `Agent`, as opposed to being separate functions; observation parsing could be in the `Task` or the `Agent` class). In general, we want to maximize separability and minimize interfaces/dependencies, so that we can easily swap out different agents for the same task, or vice versa.

## Task

In an LLM agent eval, there will usually be a `Task` class that interacts with the `Agent`. In general, the `Task` will:

- Prepare and provide the task instruction (and necessary files, functions etc) to the agent
- Parse and score the agent's output
- Update the task state accordingly (e.g. proceeds onto the next step of the task, ends the task).

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "3.4.10"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part4_llm_agents.solutions import ArithmeticTask, calculate, arithmetic_agent, agent_task, get_permitted_links, GetContentTool, WikiAgent, wiki_task, WikiAgentPrompting, WikiAgentReAct


### Exercise - Let the LLM see its entire chat history
> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 10-15 mins on this exercise.
> ```

You may have noticed that the agent performs significantly worse as a result of the fact that we decided to reset the chat history every time the agent encounters a new page. For example, it will occasionally come up with good plans and not follow through on them, since its in-context memory has been erased. We can fix this issue by letting the agent see the entirety of its chat history.

The main obstacle to allowing the agent to see its entire history is the capacity of its context window -- specifically due to the length of wikipedia articles that the agent has to retrieve in order to play the game. However, we can fix this issue by resetting **only** the outputs of the `GetContentTool` function each time the agent moves to a new page, instead of resetting the entire chat history.

When we reset this content, we should still let the agent know that Wikipedia content was output in that location, as otherwise it will confuse the agent about the `get_content` tool. You should replace the content with `"Wikipedia content was output here. Wikipedia page: {page_title}"` so that the agent knows that the `get_content` tool works as expected. 
 
Modify the `_reset_history` function in the `WikiAgentHistory` function below to accomplish this. You may also need to modify the `_handle_tool_calls` function to keep track of the previous page.

In [ ]:
@agent
def WikiAgentHistory(tools: list[Tool], game: WikiGame, verbose: bool = True):
    system_instruction =
    on_page_instruction =
    raise NotImplementedError("You need to implement the prompts for the WikiAgentHistory (copy your solutions from before).")
    async def instruction_refresh() -> None:
        nonlocal system_instruction, on_page_instruction
        raise NotImplementedError("You need to reimplement the instruction_refresh function (copy your solution from before)")

    async def _reset_history(state: AgentState, previous_page: str) -> AgentState:
        raise NotImplementedError("You need to implement the new _reset_history function")

    async def generate_reason(state: AgentState) -> AgentState:
        raise NotImplementedError("You need to implement the generate_reason function")

    async def generate_action(state: AgentState) -> AgentState:
        raise NotImplementedError("You need to implement the generate_action function")

    async def _start(state: AgentState) -> AgentState:
        raise NotImplementedError("You need to implement the _start function (copy your solution from before)")

    async def _handle_tool_calls(state: AgentState) -> AgentState:
        raise NotImplementedError("You need to reimplement the _handle_tool_calls function")

    async def execute(state: AgentState) -> AgentState:
        raise NotImplementedError("You need to implement the execute function (copy your solution from before)")

    return execute

<details><summary>Solution</summary>

```python
@agent
def WikiAgentHistory(tools: list[Tool], game: WikiGame, verbose: bool = True):
    system_instruction = ChatMessageSystem(
        content=f"You are a wikipedia-racing AI. Your goal is to reach {game.goal_page.title} by accessing links from wikipedia pages. Your current page is {game.current_page.title}."
    )

    on_page_instruction = ChatMessageUser(
        content=f"""You are currently on page: {game.current_page.title}. Make sure you start by reasoning about what steps you should take to get to the article on {game.goal_page.title}. When coming up with a strategy, make sure to pay attention to the path you have already taken, and if your current strategy doesn't seem to be working out, try something else. In case you're unsure, {game.goal_page.title} has the following summary:\n\n[Begin Summary]\n{game.get_page_summary(game.goal_page)}\n[End Summary]\n\nThe path you have taken so far is {" -> ".join(game.page_history)}.
            """
    )

    async def instruction_refresh() -> None:
        nonlocal system_instruction, on_page_instruction
        system_instruction = ChatMessageSystem(
            content=f"You are a wikipedia-racing AI. Your goal is to reach {game.goal_page.title} by accessing links from wikipedia pages. Your current page is {game.current_page.title}."
        )
        on_page_instruction = ChatMessageUser(
            content=f"""You are currently on page: {game.current_page.title}. Make sure you start by reasoning about what steps you should take to get to the article on {game.goal_page.title}. When coming up with a strategy, make sure to pay attention to the path you have already taken, and if your current strategy doesn't seem to be working out, try something else. In case you're unsure, {game.goal_page.title} has the following summary:\n\n[Begin Summary]\n{game.get_page_summary(game.goal_page)}\n[End Summary]\n\nThe path you have taken so far is {" -> ".join(game.page_history)}.
                """
        )

    async def _reset_history(state: AgentState, previous_page: str) -> AgentState:
        for message in state.messages:
            if (
                isinstance(message, ChatMessageTool)
                and message.function == "GetContentTool"
                and "Wikipedia page content for page" not in message.content
            ):
                message.content = f"Wikipedia page content for page {previous_page} was output here, but has been removed for brevity."
        return state

    async def generate_reason(state: AgentState) -> AgentState:
        state.messages.append(
            ChatMessageUser(
                content=f"""Before you decide on your next step, think carefully about what steps you should take to get to {game.goal_page.title}. When coming up with a strategy, make sure to pay attention to the path you have already taken, and if your current strategy doesn't seem to be working out, try something else."""
            )
        )
        state.output = await get_model().generate(input=state.messages, tools=tools, tool_choice="none")
        state.messages.append(state.output.message)
        return state

    async def generate_action(state: AgentState) -> AgentState:
        state.messages.append(
            ChatMessageUser(
                content=f"""Now based on your reasoning above, what action will you take to reach {game.goal_page.title}?"""
            )
        )
        state.output = await get_model().generate(input=state.messages, tools=tools, tool_choice="auto")
        state.messages.append(state.output.message)
        return state

    async def _start(state: AgentState) -> AgentState:
        state.messages.append(system_instruction)
        state.messages.append(on_page_instruction)
        return state

    async def _handle_tool_calls(state: AgentState) -> AgentState:
        messages, state.output = await execute_tools(messages=state.messages, tools=tools)
        state.messages.extend(messages)
        if state.output.message.tool_calls[0].function == "MovePageTool" and "success" in messages[-1].content.lower():
            previous_page = game.current_page.title
            await instruction_refresh()
            state = await _reset_history(state, previous_page)
        return state

    async def execute(state: AgentState) -> AgentState:
        success = False
        state = await _start(state)
        while not success:
            state = await generate_reason(state)
            state = await generate_action(state)
            if state.output.message.tool_calls:
                state = await _handle_tool_calls(state)
            if game.check_win():
                success = True
        return state

    return execute
```
</details>

Now let's test our new agent on a different path, we've proposed one below, but feel free to try it on an example path of your own choosing, and see how it performs compared to our previous `WikiAgentReAct`.

In [ ]:
game = WikiGame("Blavatnik School of Government", "Free Thai Movement")
tool_list = [GetContentTool(game), MovePageTool(game)]

@task
def wiki_task() -> Task:
    return Task(dataset=[Sample(input="", target="")], message_limit=120)

eval(
    solver=as_solver(WikiAgentReAct(tools=tool_list, game=game)),
    tasks=wiki_task(),
)

In [ ]:
game = WikiGame("Blavatnik School of Government", "Free Thai Movement")
tool_list = [GetContentTool(game), MovePageTool(game)]

@task
def wiki_task() -> Task:
    return Task(dataset=[Sample(input="", target="")], message_limit=120)

eval(
    solver=as_solver(WikiAgentHistory(tools=tool_list, game=game)),
    tasks=wiki_task(),
)

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
